# Notebook 07 - Improvement Eksperimen Berpasangan DDS dan noDDS

**Tahap penelitian:** eksperimen improvement dan evaluasi kandidat.

Notebook ini memuat hasil eksperimen tambahan Random Forest, XGBoost, dan LightGBM pada dua varian dataset dengan label dan split deal-aware yang sama. Grid kandidat diterapkan dengan protokol yang sama pada DDS (174 fitur) dan noDDS (164 fitur).

Hyperparameter serta alpha XGBoost dipilih berdasarkan AP validation. Threshold dipilih berdasarkan F1 macro validation. Kandidat terpilih lalu dievaluasi pada test untuk pelaporan; test tidak digunakan untuk memilih model atau threshold.

Grid pelatihan dapat dijalankan ulang melalui `scripts/run_paired_experiments.py`.

## 0. Setup

In [1]:
import json
from pathlib import Path
import runpy
import pandas as pd
import matplotlib.pyplot as plt

_root_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _root_candidates if (p / 'configs' / 'config.yaml').exists() and (p / 'scripts' / 'run_paired_experiments.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Repo root tidak ditemukan.')
OUT = ROOT / 'outputs' / 'results'
print('Repo:', ROOT)

Repo: D:\Bridge-Prediction


## 1. Jalankan atau muat hasil eksperimen

`RUN_TRAINING = False` memuat hasil terakhir yang tersimpan. Ubah ke `True` untuk menjalankan ulang seluruh grid pada kedua dataset; proses ini dapat memakan waktu karena melatih banyak kandidat model.

In [2]:
RUN_TRAINING = False
if RUN_TRAINING:
    runpy.run_path(str(ROOT / 'scripts' / 'run_paired_experiments.py'), run_name='__main__')
else:
    print('Memuat hasil tersimpan. Untuk melatih ulang grid, set RUN_TRAINING = True.')

results_path = OUT / 'nb06_paired_experiments_results.csv'
summary_path = OUT / 'nb06_paired_experiments_summary.json'
if not results_path.exists() or not summary_path.exists():
    raise FileNotFoundError('Artefak hasil tidak ada. Jalankan ulang dengan RUN_TRAINING = True.')
results = pd.read_csv(results_path)
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print('Baris hasil:', len(results))
print('Split:', results['split'].value_counts().to_dict())

Memuat hasil tersimpan. Untuk melatih ulang grid, set RUN_TRAINING = True.
Baris hasil: 32
Split: {'validation': 18, 'test': 14}


## 2. Kandidat dan parameter yang dipilih pada setiap dataset

In [3]:
selected_parameters = []
for dataset, values in summary['selected_parameters_by_dataset'].items():
    selected_parameters.extend([
        {'Dataset': dataset, 'Algoritma': 'XGBoost', 'Kandidat grid terpilih (AP val)': values['xgb_grid_candidate'], 'Parameter': json.dumps(values['xgb_grid_params'], ensure_ascii=False)},
        {'Dataset': dataset, 'Algoritma': 'LightGBM', 'Kandidat grid terpilih (AP val)': values['lgbm_grid_candidate'], 'Parameter': json.dumps(values['lgbm_grid_params'], ensure_ascii=False)},
        {'Dataset': dataset, 'Algoritma': 'RandomForest', 'Kandidat grid terpilih (AP val)': values['rf_grid_candidate'], 'Parameter': json.dumps(values['rf_grid_params'], ensure_ascii=False)},
        {'Dataset': dataset, 'Algoritma': 'XGBoost soft-balance', 'Kandidat grid terpilih (AP val)': f"alpha={values['best_softbalance_alpha_by_validation_ap']}", 'Parameter': 'sample_weight parsial'},
        {'Dataset': dataset, 'Algoritma': 'XGBoost threshold', 'Kandidat grid terpilih (F1 macro val)': f"threshold={values['best_threshold_by_validation_f1_macro']:.3f}", 'Parameter': 'post-hoc; model Exp F'},
    ])
display(pd.DataFrame(selected_parameters))

,Dataset,Algoritma,Kandidat grid terpilih (AP val),Parameter,Kandidat grid terpilih (F1 macro val)
0,DDS,XGBoost,deeper_more_trees,"{""n_estimators"": 500, ""max_depth"": 8, ""learnin...",NaN
1,DDS,LightGBM,more_leaves,"{""n_estimators"": 400, ""num_leaves"": 95, ""learn...",NaN
2,DDS,RandomForest,more_features,"{""n_estimators"": 400, ""max_depth"": null, ""min_...",NaN
3,DDS,XGBoost soft-balance,alpha=0.25,sample_weight parsial,NaN
4,DDS,XGBoost threshold,NaN,post-hoc; model Exp F,threshold=0.445
5,noDDS,XGBoost,default (config.yaml),"{""n_estimators"": 300, ""max_depth"": 6, ""learnin...",NaN
6,noDDS,LightGBM,deep_constrained,"{""n_estimators"": 350, ""num_leaves"": 127, ""max_...",NaN
7,noDDS,RandomForest,more_trees,"{""n_estimators"": 600, ""max_depth"": null, ""min_...",NaN
8,noDDS,XGBoost soft-balance,alpha=0.5,sample_weight parsial,NaN
9,noDDS,XGBoost threshold,NaN,post-hoc; model Exp F,threshold=0.465


## 3. Improvement pada validation dibanding baseline

Delta ditampilkan dalam percentage points (pp). Baris ini mencakup kandidat weighting dan threshold XGBoost selain retuning RF, XGBoost, dan LightGBM.

In [4]:
validation = results[results['split'] == 'validation'].copy()
base_val = validation[validation['experiment'] == 'Baseline'][
    ['dataset','algorithm','accuracy','f1_macro','average_precision']
].rename(columns={
    'accuracy':'baseline_accuracy',
    'f1_macro':'baseline_f1_macro',
    'average_precision':'baseline_ap',
})
candidates_val = validation[validation['experiment'] != 'Baseline'].merge(
    base_val, on=['dataset','algorithm'], validate='many_to_one'
)
candidates_val['delta_accuracy_pp'] = 100*(candidates_val['accuracy']-candidates_val['baseline_accuracy'])
candidates_val['delta_f1_macro_pp'] = 100*(candidates_val['f1_macro']-candidates_val['baseline_f1_macro'])
candidates_val['delta_ap_pp'] = 100*(candidates_val['average_precision']-candidates_val['baseline_ap'])
val_cols=['dataset','algorithm','experiment','baseline_accuracy','accuracy','delta_accuracy_pp',
         'baseline_f1_macro','f1_macro','delta_f1_macro_pp','baseline_ap','average_precision','delta_ap_pp']
val_view=candidates_val[val_cols].sort_values(['dataset','algorithm','average_precision'],ascending=[True,True,False])
val_format={c:'{:.2%}' for c in ['baseline_accuracy','accuracy']}
val_format.update({c:'{:.4f}' for c in ['baseline_f1_macro','f1_macro','baseline_ap','average_precision']})
val_format.update({c:'{:+.2f}' for c in ['delta_accuracy_pp','delta_f1_macro_pp','delta_ap_pp']})
display(val_view.style.format(val_format))
val_view.to_csv(OUT / 'nb07_paired_improvement_validation_comparison.csv', index=False)

,dataset,algorithm,experiment,baseline_accuracy,accuracy,delta_accuracy_pp,baseline_f1_macro,f1_macro,delta_f1_macro_pp,baseline_ap,average_precision,delta_ap_pp
4,DDS,LightGBM,Exp C: retuned,79.23%,80.51%,+1.29,0.7279,0.7354,+0.75,0.6332,0.6516,+1.84
5,DDS,RandomForest,Exp E: retuned,80.38%,81.44%,+1.06,0.7075,0.7347,+2.73,0.6094,0.6524,+4.29
1,DDS,XGBoost,Exp B: balanced penuh,81.17%,80.24%,-0.93,0.6987,0.7294,+3.07,0.6071,0.6370,+3.00
2,DDS,XGBoost,Exp F: soft-balanced alpha=0.25,81.17%,81.59%,+0.42,0.6987,0.7225,+2.38,0.6071,0.6342,+2.71
3,DDS,XGBoost,Exp F + threshold=0.445,81.17%,81.44%,+0.28,0.6987,0.7328,+3.41,0.6071,0.6342,+2.71
0,DDS,XGBoost,Exp A: retuned,81.17%,81.76%,+0.60,0.6987,0.7101,+1.14,0.6071,0.6293,+2.22
10,noDDS,LightGBM,Exp C: retuned,72.13%,73.35%,+1.22,0.6425,0.6425,+0.00,0.4590,0.4632,+0.42
11,noDDS,RandomForest,Exp E: retuned,76.53%,76.64%,+0.11,0.6295,0.6323,+0.28,0.4704,0.4742,+0.39
8,noDDS,XGBoost,Exp F: soft-balanced alpha=0.5,77.12%,75.50%,-1.62,0.5731,0.6443,+7.13,0.4665,0.4665,+0.00
9,noDDS,XGBoost,Exp F + threshold=0.465,77.12%,74.29%,-2.83,0.5731,0.6494,+7.63,0.4665,0.4665,+0.00


## 4. Evaluasi test kandidat terpilih

Model retuning dipilih per algoritma dan dataset dengan AP validation. Kandidat threshold XGBoost dipilih terpisah dengan F1 macro validation. Tabel ini melaporkan kinerja test sesudah pilihan tersebut dikunci.

In [5]:
test = results[results['split'] == 'test'].copy()
selected_test = test[test['experiment'] != 'Baseline'].copy()
test_cols=['dataset','algorithm','experiment','accuracy','f1_macro','roc_auc','average_precision']
test_format={'accuracy':'{:.2%}','f1_macro':'{:.4f}','roc_auc':'{:.4f}','average_precision':'{:.4f}'}
display(selected_test[test_cols].sort_values(['dataset','algorithm']).style.format(test_format))
selected_test[test_cols].to_csv(OUT / 'nb07_paired_improvement_test_selected.csv', index=False)

best_test = selected_test.loc[selected_test['accuracy'].idxmax()]
print('Accuracy test tertinggi: {} / {} / {} = {:.2%}'.format(
    best_test['dataset'], best_test['algorithm'], best_test['experiment'], best_test['accuracy']))

,dataset,algorithm,experiment,accuracy,f1_macro,roc_auc,average_precision
14,DDS,LightGBM,Exp C: more_leaves,80.52%,0.7361,0.8159,0.6326
12,DDS,RandomForest,Exp E: more_features,81.09%,0.7309,0.8052,0.6383
13,DDS,XGBoost,Exp A: deeper_more_trees,81.81%,0.7132,0.8098,0.6256
15,DDS,XGBoost,Exp F + threshold=0.445,81.63%,0.7364,0.8125,0.6290
30,noDDS,LightGBM,Exp C: deep_constrained,73.21%,0.6419,0.7068,0.4303
28,noDDS,RandomForest,Exp E: more_trees,75.04%,0.5994,0.7077,0.4364
29,noDDS,XGBoost,Exp A: default (config.yaml),76.00%,0.5445,0.7086,0.4386
31,noDDS,XGBoost,Exp F + threshold=0.465,73.69%,0.6391,0.7086,0.4334


Accuracy test tertinggi: DDS / XGBoost / Exp A: deeper_more_trees = 81.81%


## 5. Baseline versus improvement terpilih pada test

In [6]:
base_test = test[test['experiment'] == 'Baseline'][
    ['dataset','algorithm','accuracy','f1_macro','roc_auc','average_precision']
].rename(columns={
    'accuracy':'baseline_accuracy','f1_macro':'baseline_f1_macro',
    'roc_auc':'baseline_roc_auc','average_precision':'baseline_ap',
})
selected_by_ap = selected_test[selected_test['experiment'].str.startswith(('Exp A:','Exp C:','Exp E:'))]
paired_test = selected_by_ap.merge(base_test,on=['dataset','algorithm'],validate='one_to_one')
paired_test['delta_accuracy_pp']=100*(paired_test['accuracy']-paired_test['baseline_accuracy'])
paired_test['delta_f1_macro_pp']=100*(paired_test['f1_macro']-paired_test['baseline_f1_macro'])
paired_test['delta_ap_pp']=100*(paired_test['average_precision']-paired_test['baseline_ap'])
paired_cols=['dataset','algorithm','experiment','baseline_accuracy','accuracy','delta_accuracy_pp',
             'baseline_f1_macro','f1_macro','delta_f1_macro_pp','baseline_ap','average_precision','delta_ap_pp']
paired_view=paired_test[paired_cols].sort_values(['dataset','algorithm'])
paired_format={c:'{:.2%}' for c in ['baseline_accuracy','accuracy']}
paired_format.update({c:'{:.4f}' for c in ['baseline_f1_macro','f1_macro','baseline_ap','average_precision']})
paired_format.update({c:'{:+.2f}' for c in ['delta_accuracy_pp','delta_f1_macro_pp','delta_ap_pp']})
display(paired_view.style.format(paired_format))
paired_view.to_csv(OUT / 'nb07_paired_improvement_test_comparison.csv',index=False)

,dataset,algorithm,experiment,baseline_accuracy,accuracy,delta_accuracy_pp,baseline_f1_macro,f1_macro,delta_f1_macro_pp,baseline_ap,average_precision,delta_ap_pp
2,DDS,LightGBM,Exp C: more_leaves,79.16%,80.52%,+1.37,0.7295,0.7361,+0.66,0.6285,0.6326,+0.42
0,DDS,RandomForest,Exp E: more_features,80.32%,81.09%,+0.77,0.7095,0.7309,+2.14,0.5888,0.6383,+4.95
1,DDS,XGBoost,Exp A: deeper_more_trees,80.89%,81.81%,+0.92,0.6947,0.7132,+1.84,0.5946,0.6256,+3.10
5,noDDS,LightGBM,Exp C: deep_constrained,71.40%,73.21%,+1.81,0.6367,0.6419,+0.51,0.4180,0.4303,+1.23
3,noDDS,RandomForest,Exp E: more_trees,75.00%,75.04%,+0.04,0.5999,0.5994,-0.05,0.4356,0.4364,+0.08
4,noDDS,XGBoost,Exp A: default (config.yaml),76.00%,76.00%,+0.00,0.5445,0.5445,+0.00,0.4386,0.4386,+0.00


## 6. Ringkasan

- Seluruh kandidat improvement dibandingkan dengan baseline masing-masing pada validation.
- Model/hyperparameter dipilih dengan AP validation; threshold dipilih dengan F1 macro validation.
- Test digunakan untuk melaporkan kandidat terpilih, bukan untuk memilih pemenang.
- SHAP untuk enam baseline dibahas di Notebook 05; hasil SHAP bukan ukuran kausal.

Artefak rinci tersedia di `outputs/results/nb06_paired_experiments_results.csv` dan `outputs/results/nb06_paired_experiments_summary.json`.